Notes expérience sel: poids initial: 33 grammes

dimensions mesurées 9x9x2cm

poids retitré: 8 grammes

hauteur caméra : 13 cm

## Pipeline 2D — masques, textures, profondeur

1. **Masques** — Segmentation SLIC / variance, rognage `pad`, alignement sur la grille profondeur, `mask_combined`.
2. **Profondeur** — Depth Pro, normalisation, inversion, `depth_1` / `depth_2`, recalage `depth_2_aligned`.
3. **Analyse 2D** — Similarité RGB (ΔE), statistiques sur zone commune, visualisations matplotlib.
4. **Export** — Dernière cellule : écrit `data/processed/depth_field_v3_bundle.npz` pour le notebook **3D**.

La suite Open3D (nuages, mesh, volume) est dans `depth_field_V3_3d.ipynb`.


### Workflow actuel

1. **Imports et fonctions** — Chargement des bibliothèques (matplotlib, cv2, skimage, depth_pro, etc.) et définition des fonctions utilitaires (revert_depth, SLIC, masques variance, similarité RGB, coordonnées masque).

2. **Chemins** — Définition des images sources (BASE_NAME_1, BASE_NAME_2) et du répertoire data.

3. **Étape 1 — Segmentation** — Pipeline SLIC (CLAHE, top-hat, Otsu) sur chaque image → superpixels → masques par variance (`mask_variance`, `mask_variance_2`).

4. **Étape 2 — Cartes de profondeur** — Modèle Depth Pro, inférence sur images rognées → `depth_1`, `depth_2`. Normalisation [0,1], alignement de `depth_2` sur la grille de `depth_1` → `depth_2_aligned`. Masques rognés, redimensionnés sur la grille profondeur, **union** (`max` pixel à pixel) → `mask_combined`.

5. **Similarité RGB** — ΔE (Lab) entre les deux images, masque des zones très similaires (`mask_rgb_similar`) ∩ `mask_combined`.

6. **Rescaling** — X = moyenne(**depth_1 − depth_2_aligned**) sur `mask_rgb_similar` (avec clip numérique), puis remap linéaire de `depth_2_aligned` vers [0, X] → `depth_2_remapped`. **Inversion** proche/lointain : `depth_1` est inversé ; `depth_2_remapped` est ensuite réécrite à partir de **`depth_2`** inversé (le remap linéaire précédent n’est plus dans cette variable après cette étape).

7. **Analyse et vérification** — Moyenne (**depth_1 − depth_2_remapped**) sur `coords_xy`, contrôle des ranges, visualisations RGB et profondeur.

8. **Maximums locaux** — Pics locaux sur `depth_1` (filtrés par `mask_combined`), comparaison avec `depth_2_remapped` ; **recalcul optionnel** de `depth_2_remapped` à partir de ces pics (alternative au rescaling piloté par la similarité RGB).

9. **Export** — Enregistrement du bundle npz pour le notebook 3D.

In [1]:
import sys
print("Python utilisé:", sys.executable)
print("Premiers chemins:", sys.path[:3])

import matplotlib
matplotlib.use('tkAgg')
from matplotlib import pyplot as plt
from PIL import Image
import torch
import numpy as np
import requests
import dataclasses
from pathlib import Path
import cv2
import cv2 as cv
from skimage.segmentation import slic, mark_boundaries
import depth_pro
from depth_pro.depth_pro import create_model_and_transforms, DEFAULT_MONODEPTH_CONFIG_DICT
from transformers import GLPNImageProcessor, GLPNForDepthEstimation


Python utilisé: c:\Users\mvm\open3d_vision\.venv\Scripts\python.exe
Premiers chemins: ['C:\\Users\\mvm\\AppData\\Local\\Programs\\Python\\Python312\\python312.zip', 'C:\\Users\\mvm\\AppData\\Local\\Programs\\Python\\Python312\\DLLs', 'C:\\Users\\mvm\\AppData\\Local\\Programs\\Python\\Python312\\Lib']


In [2]:
# ========== Définitions des fonctions (utilisées dans le pipeline) ==========

def revert_depth_image(depth_image):
    """
    Inverse la profondeur de l'image : proche <-> lointain.
    depth_image : array numpy 2D (H, W), valeurs de profondeur.
    Retourne une copie avec depth_inv = depth_max - depth + depth_min (range préservé, ordre inversé).
    """
    d = np.asarray(depth_image, dtype=np.float64)
    d_min, d_max = d.min(), d.max()
    return (d_max - d + d_min).astype(depth_image.dtype if hasattr(depth_image, 'dtype') else np.float32)


def run_slic_pipeline(path, kernel, clahe):
    """Applique CLAHE, top-hat, Otsu et SLIC sur une image. Retourne un dict avec toutes les sorties."""
    img_th = cv.imread(str(path), cv.IMREAD_GRAYSCALE)
    assert img_th is not None, f"Image non trouvée: {path}"
    equalized = cv.equalizeHist(img_th)
    clahe_eq = clahe.apply(img_th)
    tophat = cv.morphologyEx(equalized, cv.MORPH_TOPHAT, kernel)
    _, th_otsu = cv.threshold(clahe_eq, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
    img_rgb = cv.cvtColor(clahe_eq, cv.COLOR_GRAY2BGR)
    segments = slic(img_rgb, n_segments=250, compactness=10, sigma=1, start_label=1)
    img_slic = (np.clip(mark_boundaries(img_rgb, segments, color=(1, 0, 0), mode="thick"), 0, 1) * 255).astype(np.uint8)
    return {"img_th": img_th, "equalized": equalized, "clahe_equalized": clahe_eq, "tophat": tophat,
            "th_otsu": th_otsu, "img_rgb": img_rgb, "segments": segments, "img_slic": img_slic}


def mask_from_variance(img_grayscale, segments, percentile=68):
    """Masque binaire : 255 = superpixels texturés (variance ≥ percentile), 0 = reste."""
    gray = np.asarray(img_grayscale, dtype=np.float64)
    labels = np.unique(segments)
    var_per_label = {lab: np.var(gray[segments == lab]) for lab in labels}
    vars_arr = np.array([var_per_label[lab] for lab in labels])
    thresh = np.percentile(vars_arr, percentile)
    interesting = set(lab for lab, v in var_per_label.items() if v >= thresh)
    return np.where(np.isin(segments, list(interesting)), 255, 0).astype(np.uint8)


def normalize_depth_map(depth):
    d_min = depth.min()
    d_max = depth.max()
    if d_max > d_min:
        return (depth - d_min) / (d_max - d_min)
    else:
        return np.zeros_like(depth)


def crop_resize_mask(mask_var, pad, h_d, w_d):
    """Rogne le masque avec pad puis le redimensionne sur (h_d, w_d)."""
    if mask_var.shape[0] > 2 * pad and mask_var.shape[1] > 2 * pad:
        mask_crop = mask_var[pad:-pad, pad:-pad]
    else:
        mask_crop = mask_var
    return cv2.resize(mask_crop, (w_d, h_d), interpolation=cv2.INTER_NEAREST)


def mask_rgb_strong_similarity(rgb_1, rgb_2, percentile_similar=8, morph_open_ksize=3):
    """
    Masque binaire : 255 = pixels où les deux images RGB sont très similaires (faible ΔE Lab).
    percentile_similar : percentile bas sur la carte ΔE (plus bas = critère plus strict).
    Retourne (masque uint8, carte delta_e, seuil utilisé).
    """
    a = np.asarray(rgb_1.convert("RGB") if hasattr(rgb_1, "convert") else rgb_1, dtype=np.uint8)
    b = np.asarray(rgb_2.convert("RGB") if hasattr(rgb_2, "convert") else rgb_2, dtype=np.uint8)
    if a.shape[:2] != b.shape[:2]:
        b = np.asarray(
            Image.fromarray(b).resize((a.shape[1], a.shape[0]), Image.Resampling.LANCZOS),
            dtype=np.uint8,
        )
    lab1 = cv2.cvtColor(a, cv2.COLOR_RGB2LAB).astype(np.float64)
    lab2 = cv2.cvtColor(b, cv2.COLOR_RGB2LAB).astype(np.float64)
    delta_e = np.sqrt(np.sum((lab1 - lab2) ** 2, axis=2))
    thresh = np.percentile(delta_e, percentile_similar)
    mask = ((delta_e <= thresh).astype(np.uint8)) * 255
    if morph_open_ksize and morph_open_ksize > 0:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (morph_open_ksize, morph_open_ksize))
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k)
    return mask, delta_e, float(thresh)


def delta_e_masked_by_common(delta_e, mask_common_uint8):
    """Retourne la carte ΔE avec NaN hors du masque commun (255 = zone conservée)."""
    m = mask_common_uint8.astype(np.uint8)
    return np.where(m == 255, delta_e, np.nan)


def pixels_coords_mask_255(mask_uint8):
    """
    Retourne les coordonnées de tous les pixels où le masque vaut 255.
    - coords_row_col : (row, col), indices numpy / imshow
    - coords_xy : (x, y) avec x = colonne, y = ligne
    """
    row, col = np.where(np.asarray(mask_uint8) == 255)
    coords_row_col = np.column_stack((row, col))
    coords_xy = np.column_stack((col, row))
    return coords_row_col, coords_xy

Création des chemins des 4 images

In [3]:
# Génération des cartes de profondeur (ordre identique à V2, pour 4 images)
# 1) Chemins des images (équivalent IMAGE_PATH de V2)
DATA_DIR = Path(r"C:\Users\mvm\open3d_vision\data")
BASE_NAME_OLD = r"pile-of-soil-top-view-isolated-on-white-G1N4P8"
BASE_NAME_1 = r"C:\Users\mvm\open3d_vision\data\photos sel\33 grammes.jpg"
BASE_NAME_2 = r"C:\Users\mvm\open3d_vision\data\photos sel\25 grammes.jpg"
paths = [
    BASE_NAME_1,
    BASE_NAME_2,
]


### Étape 1 — Segmentation objet / fond (SLIC + variance)

Pipeline appliqué en parallèle aux deux images (BASE_NAME_1 et BASE_NAME_2) : image → CLAHE → SLIC → variance → masques mask_variance (image 1) et mask_variance_2 (image 2).  
Ce masque sert ensuite à isoler l’objet dans les cartes de profondeur.

In [4]:
# Pipeline top-hat + Otsu + SLIC pour les deux images (paths[0] et paths[1]) en parallèle
kernel = cv.getStructuringElement(cv.MORPH_ELLIPSE, (25, 25))
clahe = cv.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

# Traitement des deux images (BASE_NAME_1 et BASE_NAME_2)
out_1 = run_slic_pipeline(paths[0], kernel, clahe)
out_2 = run_slic_pipeline(paths[1], kernel, clahe)

# Variables pour la suite : image 1 (compatibilité avec cellule variance)
img_th = out_1["img_th"]
equalized = out_1["equalized"]
clahe_equalized = out_1["clahe_equalized"]
tophat = out_1["tophat"]
th_tophat_otsu = out_1["th_otsu"]
img_rgb = out_1["img_rgb"]
segments = out_1["segments"]
img_slic = out_1["img_slic"]
# Image 2
img_th_2 = out_2["img_th"]
segments_2 = out_2["segments"]

# Visualisation : 2 lignes = image 1, image 2 ; 3 colonnes = Original, CLAHE+Otsu, SLIC
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for i, (out, title) in enumerate([(out_1, "Image 1 (33 g)"), (out_2, "Image 2 (25 g)")]):
    axes[i, 0].imshow(out["img_th"], cmap="gray")
    axes[i, 0].set_title(f"{title} — Original")
    axes[i, 0].axis("off")
    axes[i, 1].imshow(out["th_otsu"], cmap="gray")
    axes[i, 1].set_title(f"{title} — Otsu")
    axes[i, 1].axis("off")
    axes[i, 2].imshow(cv.cvtColor(out["img_slic"], cv.COLOR_BGR2RGB))
    axes[i, 2].set_title(f"{title} — SLIC superpixels")
    axes[i, 2].axis("off")
plt.tight_layout()
plt.show()

In [5]:
# Séparation binaire par variance des superpixels pour les deux images
percentile_var = 68  # paramètre commun aux deux images
mask_variance = mask_from_variance(img_th, segments, percentile_var)
mask_variance_2 = mask_from_variance(img_th_2, segments_2, percentile_var)

# Visualisation : 2 lignes (image 1, image 2) — original, masque, superposition
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for i, (im, mask, title) in enumerate([
    (img_th, mask_variance, "Image 1 (33 g)"),
    (img_th_2, mask_variance_2, "Image 2 (25 g)"),
]):
    axes[i, 0].imshow(im, cmap="gray")
    axes[i, 0].set_title(f"{title} — Original")
    axes[i, 0].axis("off")
    axes[i, 1].imshow(mask, cmap="gray")
    axes[i, 1].set_title(f"{title} — Masque variance (p{percentile_var})")
    axes[i, 1].axis("off")
    overlay = cv.cvtColor(im, cv.COLOR_GRAY2BGR)
    overlay[mask == 255] = [0, 180, 0]
    axes[i, 2].imshow(cv.cvtColor(overlay, cv.COLOR_BGR2RGB))
    axes[i, 2].set_title(f"{title} — Superpixels retenus")
    axes[i, 2].axis("off")
plt.tight_layout()
plt.show()

### Etape 2 — Generation des cartes de profondeur

Creation des objets image et initialisation du modele depth-pro.

In [6]:
# 2) Modèle depth_pro + chargement / transform / overwrite PIL (ordre V2 strict)
CHECKPOINT = Path(r"C:\Users\mvm\open3d_vision\ml-depth-pro\checkpoints\depth_pro_alt.pt")
config = dataclasses.replace(DEFAULT_MONODEPTH_CONFIG_DICT, checkpoint_uri=str(CHECKPOINT))
model, transform = create_model_and_transforms(config=config)
model.eval()

# Image 1 : load_rgb → transform → overwrite avec PIL (comme V2)
image_og_1, _, f_px_1 = depth_pro.load_rgb(str(paths[0]))
image_1 = transform(image_og_1)
image_1 = Image.open(paths[0]).convert("RGB")
# Image 2
image_og_2, _, f_px_2 = depth_pro.load_rgb(str(paths[1]))
image_2 = transform(image_og_2)
image_2 = Image.open(paths[1]).convert("RGB")
# # Image 3
# image_og_3, _, f_px_3 = depth_pro.load_rgb(str(paths[2]))
# image_3 = transform(image_og_3)
# image_3 = Image.open(paths[2]).convert("RGB")
# # Image 4
# image_og_4, _, f_px_4 = depth_pro.load_rgb(str(paths[3]))
# image_4 = transform(image_og_4)
# image_4 = Image.open(paths[3]).convert("RGB")


Inférence des images pour cartes de profondeur

In [7]:
# 5) Rognage pad 
pad = 16
image_cropped_1 = image_1.crop((pad, pad, image_1.width - pad, image_1.height - pad))
image_cropped_2 = image_2.crop((pad, pad, image_2.width - pad, image_2.height - pad))
# image_cropped_3 = image_3.crop((pad, pad, image_3.width - pad, image_3.height - pad))
# image_cropped_4 = image_4.crop((pad, pad, image_4.width - pad, image_4.height - pad))


In [8]:
# 3) Inférence depth_pro pour chaque image (ordre V2)
# On doit passer à model.infer le tenseur transformé, pas l'image PIL !
prediction_1 = model.infer(transform(image_cropped_1), f_px=f_px_1)
depth_1 = prediction_1["depth"].squeeze().cpu().numpy()
prediction_2 = model.infer(transform(image_cropped_2), f_px=f_px_2)
depth_2 = prediction_2["depth"].squeeze().cpu().numpy()
# prediction_3 = model.infer(transform(image_og_3), f_px=f_px_3)
# depth_3 = prediction_3["depth"].squeeze().cpu().numpy()
# prediction_4 = model.infer(transform(image_og_4), f_px=f_px_4)
# depth_4 = prediction_4["depth"].squeeze().cpu().numpy()


Normalisation des cartes de profondeur sur un range [0,1]

In [9]:
# Normalisation des cartes de profondeur (min=0, max=1)
depth_1 = normalize_depth_map(depth_1)
depth_2 = normalize_depth_map(depth_2)
# depth_3, depth_4 : désactivés (images 3 et 4 non chargées)
# depth_3 = normalize_depth_map(depth_3)
# depth_4 = normalize_depth_map(depth_4)

In [10]:
# Decoupage + superposition des masques sur la grille profondeur de l'image 1
pad = 16  # meme valeur que le rognage des images RGB
h_d1, w_d1 = depth_1.shape

# Base commune: la 2e depth map est alignee sur la grille de depth_1
if depth_2.shape != (h_d1, w_d1):
    depth_2_aligned = cv2.resize(
        depth_2.astype(np.float32), (w_d1, h_d1), interpolation=cv2.INTER_LINEAR
    ).astype(np.float64)
else:
    depth_2_aligned = np.asarray(depth_2, dtype=np.float64)

# Masques objets alignes sur la meme grille
mask_obj = crop_resize_mask(mask_variance, pad, h_d1, w_d1)
mask_obj_2 = crop_resize_mask(mask_variance_2, pad, h_d1, w_d1)
mask_combined = np.maximum(mask_obj, mask_obj_2)

# Visualisation du decoupage et de la superposition
rgb1_show = np.asarray(image_cropped_1.convert("RGB"), dtype=np.uint8)
if rgb1_show.shape[:2] != (h_d1, w_d1):
    rgb1_show = np.asarray(
        Image.fromarray(rgb1_show).resize((w_d1, h_d1), Image.Resampling.LANCZOS),
        dtype=np.uint8,
    )

overlay_1 = rgb1_show.copy()
overlay_1[mask_obj == 255] = [255, 70, 70]
overlay_2 = rgb1_show.copy()
overlay_2[mask_obj_2 == 255] = [70, 255, 70]
overlay_union = rgb1_show.copy()
overlay_union[mask_combined == 255] = [255, 220, 0]

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes[0, 0].imshow(mask_obj, cmap="gray")
axes[0, 0].set_title("mask_obj (image 1 aligne)")
axes[0, 0].axis("off")
axes[0, 1].imshow(mask_obj_2, cmap="gray")
axes[0, 1].set_title("mask_obj_2 (image 2 aligne)")
axes[0, 1].axis("off")
axes[0, 2].imshow(mask_combined, cmap="gray")
axes[0, 2].set_title("mask_combined (union)")
axes[0, 2].axis("off")

axes[1, 0].imshow(overlay_1)
axes[1, 0].set_title("Superposition mask_obj")
axes[1, 0].axis("off")
axes[1, 1].imshow(overlay_2)
axes[1, 1].set_title("Superposition mask_obj_2")
axes[1, 1].axis("off")
axes[1, 2].imshow(overlay_union)
axes[1, 2].set_title("Superposition union")
axes[1, 2].axis("off")

plt.tight_layout()
plt.show()

In [11]:
# RGB rognées comme le reste du pipeline (même pad que depth)
P_SIM_RGB = 25  # abaisser pour masque plus strict (moins de pixels)

mask_rgb_similar_raw, delta_e_map, thresh_de = mask_rgb_strong_similarity(
    image_cropped_1, image_cropped_2, percentile_similar=P_SIM_RGB
)

# Aligner mask_combined (taille depth) sur les RGB rognées : similarité uniquement DANS le masque commun
h_sim, w_sim = mask_rgb_similar_raw.shape[:2]
mask_combined_for_rgb = cv2.resize(
    mask_combined, (w_sim, h_sim), interpolation=cv2.INTER_NEAREST
)
delta_e_in_common = delta_e_masked_by_common(delta_e_map, mask_combined_for_rgb)
# Zones très similaires (binaire) ∩ objet SLIC commun
mask_rgb_similar = delta_e_masked_by_common(
    mask_rgb_similar_raw, mask_combined_for_rgb
)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes[0, 0].imshow(image_cropped_1)
axes[0, 0].set_title("Image 1 — RGB")
axes[0, 0].axis("off")
axes[0, 1].imshow(image_cropped_2)
axes[0, 1].set_title("Image 2 — RGB")
axes[0, 1].axis("off")
im_de = axes[0, 2].imshow(delta_e_in_common, cmap="magma")
axes[0, 2].set_title("ΔE (Lab) — uniquement dans mask_combined (↓ = plus similaire)")
axes[0, 2].axis("off")
plt.colorbar(im_de, ax=axes[0, 2], fraction=0.046, label="ΔE")
axes[1, 0].imshow(mask_combined_for_rgb, cmap="gray")
axes[1, 0].set_title("mask_combined (redim. → RGB)")
axes[1, 0].axis("off")
axes[1, 1].imshow(mask_rgb_similar_raw, cmap="gray")
axes[1, 1].set_title(f"Très similaires (ΔE ≤ p{P_SIM_RGB} ≈ {thresh_de:.1f}), plein cadre")
axes[1, 1].axis("off")
axes[1, 2].imshow(mask_rgb_similar, cmap="gray")
axes[1, 2].set_title("Très similaires ∩ mask_combined")
axes[1, 2].axis("off")
plt.tight_layout()
plt.show()

In [12]:
# Rescaling depth_2_aligned → range [0, X] avec X < 1
# X = moyenne des |depth_1 - depth_2| pour les pixels dans mask_rgb_similar
# On utilise les coordonnées exactes (coords_xy) pour garantir la même correspondance depth/masque.
h_d1, w_d1 = depth_1.shape
if depth_2_aligned.shape != (h_d1, w_d1):
    depth_2_aligned = cv2.resize(
        depth_2_aligned.astype(np.float32), (w_d1, h_d1), interpolation=cv2.INTER_LINEAR
    ).astype(np.float64)
else:
    depth_2_aligned = np.asarray(depth_2_aligned, dtype=np.float64)

# Coordonnées des pixels similaires (calculées ici pour éviter les problèmes d'ordre d'exécution)
# Procédure détaillée pour calculer X, la moyenne de la différence absolue de profondeur :
# 1. On extrait les coordonnées (x, y) de tous les pixels qui appartiennent au masque de similarité RGB (valeur 255 dans mask_rgb_similar).
_, coords_xy_local = pixels_coords_mask_255(mask_rgb_similar)  # coords_xy_local shape: (N, 2)

# 2. Séparation des coordonnées en deux tableaux pour les x et les y.
x_coords = coords_xy_local[:, 0].astype(np.intp)
y_coords = coords_xy_local[:, 1].astype(np.intp)

# 3. Vérification : seules les coordonnées dans les limites valides de l'image sont conservées
# (cela protège contre des dépassements dus à des erreurs ou des masques mal alignés).
in_bounds = (y_coords >= 0) & (y_coords < h_d1) & (x_coords >= 0) & (x_coords < w_d1)
x_coords = x_coords[in_bounds]
y_coords = y_coords[in_bounds]

# 4. Si l'on a au moins un pixel valide :
if len(x_coords) > 0:
    # — on extrait les valeurs de profondeur pour ces coordonnées dans depth_1 et depth_2_aligned
    d1_at_pts = depth_1[y_coords, x_coords].astype(np.float64)
    d2_at_pts = depth_2_aligned[y_coords, x_coords].astype(np.float64)
    # — on calcule la différence absolue entre ces deux profondeurs pixel à pixel,
    #   puis on fait la moyenne sur tous les pixels retenus pour obtenir X.
    X = float(np.mean(d1_at_pts - d2_at_pts))
    # — on s'assure que X reste dans un intervalle raisonnable (0.01 à 0.99) pour éviter des cas aberrants.
    X = np.clip(X, 0.01, 0.99)
    # — on enregistre aussi le nombre total de pixels utilisés pour le calcul.
    n_pts = len(x_coords)
else:
    # 5. Cas où aucun pixel n'est valide : on lève une erreur pour signaler ce cas.
    raise ValueError("Aucun pixel valide dans mask_rgb_similar pour le recalage de profondeur : impossible de calculer X.")

# Rescaling depth_2_aligned vers [0, X]
d2_min, d2_max = float(np.min(depth_2_aligned)), float(np.max(depth_2_aligned))
if (d2_max - d2_min) > 1e-12:
    depth_2_remapped = (depth_2_aligned - d2_min) / (d2_max - d2_min) * X
else:
    depth_2_remapped = np.full_like(depth_2_aligned, X / 2)
print(f"X = {X:.6f} (moyenne |depth_1 - depth_2| sur {n_pts} pixels de mask_rgb_similar)")
print(f"depth_2_remapped rescalé de [{d2_min:.6f}, {d2_max:.6f}] → [0, {X:.6f}]")


X = 0.170524 (moyenne |depth_1 - depth_2| sur 65161 pixels de mask_rgb_similar)
depth_2_remapped rescalé de [0.000000, 1.000000] → [0, 0.170524]


In [13]:
print(np.mean(d1_at_pts))
print(np.mean(d2_at_pts))

0.4583966060128527
0.28787275401351153


In [14]:
# 6) Revert depth 
depth_1 = revert_depth_image(depth_1)
depth_2_remapped = revert_depth_image(depth_2)
depth_2_aligned = revert_depth_image(depth_2)

# depth_3, depth_4 : désactivés
# depth_3 = revert_depth_image(depth_3)
# depth_4 = revert_depth_image(depth_4)

### Etape 1 — Decoupage + superposition des masques

Les masques SLIC (`mask_variance`, `mask_variance_2`) sont rognees avec le meme `pad` que les images, redimensionnes sur la grille profondeur, puis combines (`mask_combined`).
Cette etape fixe la zone objet utilisee ensuite pour l'adaptation de range et la 3D.

In [15]:
# Comparaison des deux images : RGB et profondeur (plus de flip / soustraction)
fig, ax = plt.subplots(2, 2, figsize=(12, 10))
ax[0, 0].imshow(image_cropped_1)
ax[0, 0].set_title("Image 1 (33 g) — RGB")
ax[0, 0].axis("off")
ax[0, 1].imshow(image_cropped_2)
ax[0, 1].set_title("Image 2 (25 g) — RGB")
ax[0, 1].axis("off")
im0 = ax[1, 0].imshow(depth_1, cmap='plasma')
ax[1, 0].set_title("Image 1 — Profondeur")
ax[1, 0].axis("off")
plt.colorbar(im0, ax=ax[1, 0], fraction=0.046)
im1 = ax[1, 1].imshow(depth_2_remapped, cmap='plasma')
ax[1, 1].set_title("Image 2 — Profondeur")
ax[1, 1].axis("off")
plt.colorbar(im1, ax=ax[1, 1], fraction=0.046)
plt.tight_layout()
plt.show()


In [16]:
coords_row_col, coords_xy = pixels_coords_mask_255(mask_rgb_similar)
print(mask_rgb_similar.shape)
print(f"Nombre de pixels à 255 : {len(coords_row_col)}")
print(f"Exemple (5 premiers) row,col : {coords_row_col[:5]}")
print(f"Exemple (5 premiers) x,y   : {coords_xy[:5]}")

(1811, 1350)
Nombre de pixels à 255 : 65161
Exemple (5 premiers) row,col : [[257 660]
 [258 660]
 [259 660]
 [260 659]
 [260 660]]
Exemple (5 premiers) x,y   : [[660 257]
 [660 258]
 [660 259]
 [659 260]
 [660 260]]


In [17]:
# Moyenne de (depth_1 - depth_2) sur les pixels de coords_xy
# Convention : coords_xy = (x, y) avec x = colonne, y = ligne → indexation depth[y, x]
h1, w1 = depth_1.shape
if depth_2.shape != (h1, w1):
    depth_2_aligned = cv2.resize(
        depth_2.astype(np.float32), (w1, h1), interpolation=cv2.INTER_LINEAR
    )
else:
    depth_2_aligned = np.asarray(depth_2, dtype=np.float64)

x_coords = coords_xy[:, 0]
y_coords = coords_xy[:, 1]
in_bounds = (
    (y_coords >= 0) & (y_coords < h1) & (x_coords >= 0) & (x_coords < w1)
)
if not np.all(in_bounds):
    print(f"Avertissement : {int(np.sum(~in_bounds))} coordonnée(s) hors limites — ignorées.")
x_coords = x_coords[in_bounds]
y_coords = y_coords[in_bounds]

d1_at_pts = depth_1[y_coords, x_coords].astype(np.float64)
d2_at_pts = depth_2_remapped[y_coords, x_coords].astype(np.float64)
mean_diff_depth_1_minus_2 = float(np.mean(d1_at_pts - d2_at_pts))
print(f"Moyenne (depth_1 - depth_2) sur coords_xy : {mean_diff_depth_1_minus_2}")


Moyenne (depth_1 - depth_2) sur coords_xy : -0.17052385200974626


In [18]:
# Recalcul de depth_2_remapped à partir des maximums locaux (à la place des pixels de similitude RGB)
# Utilise coords_peaks de la cellule précédente — même logique de rescaling que mask_rgb_similar
h_d1, w_d1 = depth_1.shape
if depth_2_remapped.shape != (h_d1, w_d1):
    depth_2_remapped = cv2.resize(
        depth_2_remapped.astype(np.float32), (w_d1, h_d1), interpolation=cv2.INTER_LINEAR
    ).astype(np.float64)
else:
    depth_2_remapped = np.asarray(depth_2_remapped, dtype=np.float64)

# Revert pour être cohérent avec depth_1 (même espace proche/lointain)
depth_2_aligned_rev = revert_depth_image(depth_2_remapped)
d1_at_peaks = depth_1[rows, cols].astype(np.float64)
d2_at_peaks = depth_2_aligned_rev[rows, cols].astype(np.float64)
X_peaks = float(np.mean(d1_at_peaks - d2_at_peaks))
X_peaks = np.clip(X_peaks, 0.01, 0.99)
n_peaks = len(rows)

# Rescaling depth_2_aligned_rev → [0, X_peaks]
d2_min, d2_max = float(np.min(depth_2_aligned_rev)), float(np.max(depth_2_aligned_rev))
if (d2_max - d2_min) > 1e-12:
    depth_2_remapped = (depth_2_aligned_rev - d2_min) / (d2_max - d2_min) * X_peaks
else:
    depth_2_remapped = np.full_like(depth_2_aligned_rev, X_peaks / 2)

print(f"X (max locaux) = {X_peaks:.6f} (moyenne depth_1 − depth_2 sur {n_peaks} pics)")
print(f"depth_2_remapped recalculé : rescaling [{d2_min:.6f}, {d2_max:.6f}] → [0, {X_peaks:.6f}]")

NameError: name 'rows' is not defined

In [ ]:
# Etape 3 — Vérification des ranges (depth_2_aligned a été rescalé en [0, X] dans la cellule précédente)
d1 = np.asarray(depth_1, dtype=np.float64)
d2a = np.asarray(depth_2_remapped, dtype=np.float64)

print(f"Range depth_1 (global): [{float(np.nanmin(d1)):.6f}, {float(np.nanmax(d1)):.6f}]")
print(f"Range depth_2_remapped (rescalé [0, X]): [{float(np.nanmin(d2a)):.6f}, {float(np.nanmax(d2a)):.6f}]")


Range depth_1 (global): [0.000000, 1.000000]
Range depth_2_remapped (rescalé [0, X]): [0.000000, 0.704605]


In [ ]:
# depth_1, depth_2_aligned et RGB associées (même grille que les cartes de profondeur)
h_d, w_d = depth_1.shape
rgb1 = np.asarray(image_cropped_1)
if rgb1.dtype != np.uint8:
    rgb1 = (np.clip(rgb1, 0, 1) * 255).astype(np.uint8)
if rgb1.shape[0] != h_d or rgb1.shape[1] != w_d:
    rgb1 = np.asarray(
        Image.fromarray(rgb1).resize((w_d, h_d), Image.Resampling.LANCZOS),
        dtype=np.uint8,
    )

rgb2 = np.asarray(image_cropped_2)
if rgb2.dtype != np.uint8:
    rgb2 = (np.clip(rgb2, 0, 1) * 255).astype(np.uint8)
if rgb2.shape[0] != h_d or rgb2.shape[1] != w_d:
    rgb2 = np.asarray(
        Image.fromarray(rgb2).resize((w_d, h_d), Image.Resampling.LANCZOS),
        dtype=np.uint8,
    )

d2_show = np.asarray(depth_2_remapped, dtype=np.float64)
if d2_show.shape != (h_d, w_d):
    d2_show = cv2.resize(
        d2_show.astype(np.float32), (w_d, h_d), interpolation=cv2.INTER_LINEAR
    )

# Amélioration de la disposition et de la taille des images
# Nouvelle disposition : 2x3 grille, agrandissement des images RGB et profondeur side-by-side, puis la différence sur toute la largeur

fig = plt.figure(figsize=(18, 12))  # Taille augmentée

# Grille 2x3: [RGB1 | depth1 | filler][RGB2 | depth2_aligned | filler], puis la différence en dessous sur 1x3
from matplotlib import gridspec
gs = gridspec.GridSpec(3, 3, height_ratios=[1.2, 1.2, 1.2], hspace=0.30, wspace=0.20)

# Premières lignes : chaque image RGB à gauche, profondeur à droite
ax00 = fig.add_subplot(gs[0, 0])  # RGB1
ax01 = fig.add_subplot(gs[0, 1])  # depth1
ax10 = fig.add_subplot(gs[1, 0])  # RGB2
ax11 = fig.add_subplot(gs[1, 1])  # depth2_aligned

# Étalement sur 2 colonnes et 3ème ligne complète pour la différence
ax_diff = fig.add_subplot(gs[2, :2])  # Différence sur deux colonnes

# Si on veut un panneau libre pour d'autres visus ou du texte :
# ax_filler1 = fig.add_subplot(gs[0, 2])
# ax_filler2 = fig.add_subplot(gs[1, 2])
# for ax in [ax_filler1, ax_filler2]: ax.axis("off")

# Calcul des bornes pour depth2_aligned
d2_lo, d2_hi = float(np.min(d2_show)), float(np.max(d2_show))
d1_f = np.asarray(depth_1, dtype=np.float64)
diff_depth = d1_f - d2_show
abs_max = float(np.max(np.abs(diff_depth))) if diff_depth.size else 0.0
if abs_max < 1e-12:
    abs_max = 1.0

# Affichages
ax00.imshow(rgb1)
ax00.set_title("Image 1 (33 g) — RGB", fontsize=14)
ax00.axis("off")

im0 = ax01.imshow(depth_1, cmap="plasma")
ax01.set_title("depth_1", fontsize=14)
ax01.axis("off")
cbar0 = fig.colorbar(im0, ax=ax01, fraction=0.046)
cbar0.ax.tick_params(labelsize=10)

ax10.imshow(rgb2)
ax10.set_title("Image 2 (25 g) — RGB", fontsize=14)
ax10.axis("off")

im1 = ax11.imshow(d2_show, cmap="plasma", vmin=0, vmax=1)
ax11.set_title(
    f"depth_2_aligned — échelle couleur [0, 1]\nvaleurs [{d2_lo:.4g}, {d2_hi:.4g}]", fontsize=14
)
ax11.axis("off")
cbar1 = fig.colorbar(im1, ax=ax11, fraction=0.046, label="intensité colormap (0–1)")
cbar1.ax.tick_params(labelsize=10)

im_d = ax_diff.imshow(diff_depth, cmap="coolwarm", vmin=-abs_max, vmax=abs_max)
ax_diff.set_title(
    f"depth_1 − depth_2_aligned (coolwarm, ±{abs_max:.4g}) — "
    f"moyenne={float(np.mean(diff_depth)):.4g}, écart-type={float(np.std(diff_depth)):.4g}",
    fontsize=14
)
ax_diff.axis("off")
cbar_d = fig.colorbar(im_d, ax=ax_diff, fraction=0.08, pad=0.02, label="Δ profondeur")
cbar_d.ax.tick_params(labelsize=10)

plt.tight_layout()
plt.show()

C:\Users\mvm\AppData\Local\Temp\ipykernel_23700\2748686117.py:91: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


In [ ]:
# Etape 4 — Preparation des cartes masquées pour la 3D
# Cette cellule suppose que depth_2_aligned a deja ete calculee puis remappee (etape 3).
if "depth_2_remapped" not in locals():
    raise ValueError("depth_2_remapped manquant: execute l'etape 3 avant la 3D.")

h_d1, w_d1 = depth_1.shape
depth_2_remapped = np.asarray(depth_2_remapped, dtype=np.float64)
if depth_2_remapped.shape != (h_d1, w_d1):
    depth_2_remapped = cv2.resize(
        depth_2_remapped.astype(np.float32), (w_d1, h_d1), interpolation=cv2.INTER_LINEAR
    ).astype(np.float64)

depth_1_masked = np.where(mask_combined == 255, depth_1.astype(np.float64), np.nan)
depth_2_masked = np.where(mask_combined == 255, depth_2_remapped, np.nan)

# Normalisation pour l'affichage uniquement
d1_min, d1_max = np.nanmin(depth_1), np.nanmax(depth_1)
depth_1_norm = (depth_1 - d1_min) / (d1_max - d1_min + 1e-8) if d1_max > d1_min else np.zeros_like(depth_1)
d2_min, d2_max = np.nanmin(depth_2_remapped), np.nanmax(depth_2_remapped)
depth_2_norm = (
    (depth_2_remapped - d2_min) / (d2_max - d2_min + 1e-8)
    if d2_max > d2_min else np.zeros_like(depth_2_remapped)
)

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes[0, 0].imshow(depth_1_norm, cmap="plasma")
axes[0, 0].set_title("Image 1 — depth_1 (norm affichee)")
axes[0, 0].axis("off")
axes[0, 1].imshow(mask_combined, cmap="gray")
axes[0, 1].set_title("Masque commun (union)")
axes[0, 1].axis("off")
axes[0, 2].imshow(depth_1_masked, cmap="plasma")
axes[0, 2].set_title("Image 1 — depth_1 masquee")
axes[0, 2].axis("off")

axes[1, 0].imshow(depth_2_norm, cmap="plasma")
axes[1, 0].set_title("Image 2 — depth_2_remapped (norm affichee)")
axes[1, 0].axis("off")
axes[1, 1].imshow(mask_obj_2, cmap="gray")
axes[1, 1].set_title("Image 2 — masque depth_2_remapped aligne")
axes[1, 1].axis("off")
axes[1, 2].imshow(depth_2_masked, cmap="plasma")
axes[1, 2].set_title("Image 2 — depth_2_remapped masquee")
axes[1, 2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# depth_1, depth_2_aligned et RGB associées (même grille que les cartes de profondeur)
h_d, w_d = depth_1.shape
rgb1 = np.asarray(image_cropped_1)
if rgb1.dtype != np.uint8:
    rgb1 = (np.clip(rgb1, 0, 1) * 255).astype(np.uint8)
if rgb1.shape[0] != h_d or rgb1.shape[1] != w_d:
    rgb1 = np.asarray(
        Image.fromarray(rgb1).resize((w_d, h_d), Image.Resampling.LANCZOS),
        dtype=np.uint8,
    )

rgb2 = np.asarray(image_cropped_2)
if rgb2.dtype != np.uint8:
    rgb2 = (np.clip(rgb2, 0, 1) * 255).astype(np.uint8)
if rgb2.shape[0] != h_d or rgb2.shape[1] != w_d:
    rgb2 = np.asarray(
        Image.fromarray(rgb2).resize((w_d, h_d), Image.Resampling.LANCZOS),
        dtype=np.uint8,
    )

d2_show = np.asarray(depth_2_remapped, dtype=np.float64)
if d2_show.shape != (h_d, w_d):
    d2_show = cv2.resize(
        d2_show.astype(np.float32), (w_d, h_d), interpolation=cv2.INTER_LINEAR
    )

# Appliquer le masque commun
depth_1_masked_for_show = np.where(mask_combined == 255, depth_1.astype(np.float64), np.nan)
d2_show_masked = np.where(mask_combined == 255, d2_show, np.nan)

# Amélioration de la disposition et de la taille des images
# Nouvelle disposition : 2x3 grille, agrandissement des images RGB et profondeur side-by-side, puis la différence sur toute la largeur

fig = plt.figure(figsize=(18, 12))  # Taille augmentée

# Grille 2x3: [RGB1 | depth1 | filler][RGB2 | depth2_aligned | filler], puis la différence en dessous sur 1x3
from matplotlib import gridspec
gs = gridspec.GridSpec(3, 3, height_ratios=[1.2, 1.2, 1.2], hspace=0.30, wspace=0.20)

# Premières lignes : chaque image RGB à gauche, profondeur à droite
ax00 = fig.add_subplot(gs[0, 0])  # RGB1
ax01 = fig.add_subplot(gs[0, 1])  # depth1 masked
ax10 = fig.add_subplot(gs[1, 0])  # RGB2
ax11 = fig.add_subplot(gs[1, 1])  # depth2_aligned masked

# Étalement sur 2 colonnes et 3ème ligne complète pour la différence
ax_diff = fig.add_subplot(gs[2, :2])  # Différence sur deux colonnes

# Calcul des bornes pour depth2_aligned masked
finite_mask = np.isfinite(d2_show_masked)
if np.any(finite_mask):
    d2_lo, d2_hi = float(np.nanmin(d2_show_masked)), float(np.nanmax(d2_show_masked))
else:
    d2_lo, d2_hi = 0.0, 1.0
d1_f_masked = np.asarray(depth_1_masked_for_show, dtype=np.float64)
diff_depth_masked = d1_f_masked - d2_show_masked
abs_max = float(np.nanmax(np.abs(diff_depth_masked))) if np.any(np.isfinite(diff_depth_masked)) else 1.0
if abs_max < 1e-12:
    abs_max = 1.0

# Affichages
ax00.imshow(rgb1)
ax00.set_title("Image 1 (33 g) — RGB", fontsize=14)
ax00.axis("off")

im0 = ax01.imshow(depth_1_masked_for_show, cmap="plasma")
ax01.set_title("depth_1 (après masque commun)", fontsize=14)
ax01.axis("off")
cbar0 = fig.colorbar(im0, ax=ax01, fraction=0.046)
cbar0.ax.tick_params(labelsize=10)

ax10.imshow(rgb2)
ax10.set_title("Image 2 (25 g) — RGB", fontsize=14)
ax10.axis("off")

im1 = ax11.imshow(d2_show_masked, cmap="plasma", vmin=0, vmax=1)
ax11.set_title(
    f"depth_2_aligned (après masque commun)\n[0, 1], valeurs [{d2_lo:.4g}, {d2_hi:.4g}]", fontsize=14
)
ax11.axis("off")
cbar1 = fig.colorbar(im1, ax=ax11, fraction=0.046, label="intensité colormap (0–1)")
cbar1.ax.tick_params(labelsize=10)

im_d = ax_diff.imshow(diff_depth_masked, cmap="coolwarm", vmin=-abs_max, vmax=abs_max)
mean_diff = float(np.nanmean(diff_depth_masked)) if np.any(np.isfinite(diff_depth_masked)) else 0.0
std_diff = float(np.nanstd(diff_depth_masked)) if np.any(np.isfinite(diff_depth_masked)) else 0.0
ax_diff.set_title(
    f"moyenne={mean_diff:.4g}, écart-type={std_diff:.4g}",
    fontsize=14
)
ax_diff.axis("off")
cbar_d = fig.colorbar(im_d, ax=ax_diff, fraction=0.08, pad=0.02, label="Δ profondeur")
cbar_d.ax.tick_params(labelsize=10)

plt.tight_layout()
plt.show()

C:\Users\mvm\AppData\Local\Temp\ipykernel_27604\3031672517.py:95: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


In [ ]:
# Cartes de profondeur : comparaison et difference avec depth_2_aligned
h1, w1 = depth_1.shape
depth_2_remapped = np.asarray(depth_2_remapped, dtype=np.float64)
if depth_2_remapped.shape != (h1, w1):
    depth_2_remapped = cv2.resize(
        depth_2_remapped.astype(np.float32), (w1, h1), interpolation=cv2.INTER_LINEAR
    ).astype(np.float64)

depth_1_f = depth_1.astype(np.float64)
depth_2_f = depth_2_remapped
diff_map = depth_1_f - depth_2_f
diff_map_masked = np.where(mask_combined == 255, diff_map, np.nan)

fig, axes = plt.subplots(3, 3, figsize=(14, 12))
v_abs = np.nanmax(np.abs(diff_map)) if np.any(np.isfinite(diff_map)) else 1.0
if v_abs < 1e-12:
    v_abs = 1.0
v_max_d = max(np.nanmax(depth_1_f), np.nanmax(depth_2_f)) if np.any(np.isfinite(depth_1_f)) else 1.0

# Ligne 1 : RGB + difference brute
axes[0, 0].imshow(image_cropped_1)
axes[0, 0].set_title("Image 1 (33 g) — RGB")
axes[0, 0].axis("off")
axes[0, 1].imshow(image_cropped_2)
axes[0, 1].set_title("Image 2 (25 g) — RGB")
axes[0, 1].axis("off")
im0 = axes[0, 2].imshow(diff_map, cmap="coolwarm", vmin=-v_abs, vmax=v_abs)
axes[0, 2].set_title("Difference brute: depth_1 - depth_2_aligned")
axes[0, 2].axis("off")
plt.colorbar(im0, ax=axes[0, 2], fraction=0.046, label="Delta profondeur")

# Ligne 2 : profondeurs masquees + difference masquee
depth_1_display = np.where(mask_combined == 255, depth_1_f, np.nan)
depth_2_display = np.where(mask_combined == 255, depth_2_f, np.nan)
im_d1 = axes[1, 0].imshow(depth_1_display, cmap="plasma", vmin=0, vmax=v_max_d)
axes[1, 0].set_title("depth_1 (masque commun)")
axes[1, 0].axis("off")
plt.colorbar(im_d1, ax=axes[1, 0], fraction=0.046, label="Profondeur")
im_d2 = axes[1, 1].imshow(depth_2_display, cmap="plasma", vmin=0, vmax=v_max_d)
axes[1, 1].set_title("depth_2_aligned (masque commun)")
axes[1, 1].axis("off")
plt.colorbar(im_d2, ax=axes[1, 1], fraction=0.046, label="Profondeur")
im1 = axes[1, 2].imshow(diff_map_masked, cmap="coolwarm", vmin=-v_abs, vmax=v_abs)
axes[1, 2].set_title("Difference masquee")
axes[1, 2].axis("off")
plt.colorbar(im1, ax=axes[1, 2], fraction=0.046, label="Delta profondeur")

# Ligne 3 : masque commun
axes[2, 0].imshow(mask_combined, cmap="gray")
axes[2, 0].set_title("Masque commun (union)")
axes[2, 0].axis("off")
axes[2, 1].axis("off")
axes[2, 2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Différence par pixel des deux cartes de profondeur (masquées), puis somme
diff_pixels = depth_1_masked - depth_2_masked
somme_diff = np.nansum(diff_pixels)
filtered_diff_pixels = diff_pixels[np.isfinite(diff_pixels)]
filtered_diff_pixels.sum()


NameError: name 'depth_1_masked' is not defined

In [ ]:
# Export pour depth_field_V3_3d.ipynb (nuages, mesh, volume)
from pathlib import Path
import numpy as np
from PIL import Image

EXPORT_DIR = Path("C:/Users/mvm/open3d_vision/data/processed")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_PATH = EXPORT_DIR / "depth_field_v3_bundle.npz"

h_d, w_d = depth_1.shape
rgb1 = np.asarray(image_cropped_1)
if rgb1.dtype != np.uint8:
    rgb1 = (np.clip(rgb1, 0, 1) * 255).astype(np.uint8)
if rgb1.shape[0] != h_d or rgb1.shape[1] != w_d:
    rgb1 = np.asarray(
        Image.fromarray(rgb1).resize((w_d, h_d), Image.Resampling.LANCZOS),
        dtype=np.uint8,
    )
rgb2 = np.asarray(image_cropped_2)
if rgb2.dtype != np.uint8:
    rgb2 = (np.clip(rgb2, 0, 1) * 255).astype(np.uint8)
if rgb2.shape[0] != h_d or rgb2.shape[1] != w_d:
    rgb2 = np.asarray(
        Image.fromarray(rgb2).resize((w_d, h_d), Image.Resampling.LANCZOS),
        dtype=np.uint8,
    )

depth_2_a = np.asarray(depth_2_remapped, dtype=np.float64)
if depth_2_a.shape != (h_d, w_d):
    import cv2
    depth_2_a = cv2.resize(
        depth_2_a.astype(np.float32), (w_d, h_d), interpolation=cv2.INTER_LINEAR
    ).astype(np.float64)

np.savez_compressed(
    EXPORT_PATH,
    depth_1=np.asarray(depth_1, dtype=np.float64),
    depth_2_remapped=depth_2_a,
    mask_combined=np.asarray(mask_combined, dtype=np.uint8),
    rgb1=rgb1,
    rgb2=rgb2,
    pad=np.int32(pad),
)
print(f"Export OK : {EXPORT_PATH}")
print(f"  depth {h_d}x{w_d}, pad={pad}")

# Ajout de l'export en jpg à côté du .npz
export_rgb1_jpg = EXPORT_DIR / "depth_field_v3_rgb1.jpg"
export_rgb2_jpg = EXPORT_DIR / "depth_field_v3_rgb2.jpg"

# Export depth maps en jpg (en les normalisant dans l'intervalle [0, 255] et cast en uint8)
def depth_to_uint8_for_jpg(depth):
    depth_min = np.nanmin(depth)
    depth_max = np.nanmax(depth)
    if np.isnan(depth_min) or np.isnan(depth_max) or depth_max == depth_min:
        out = np.zeros_like(depth, dtype=np.uint8)
    else:
        out = 255 * (depth - depth_min) / (depth_max - depth_min)
        out = np.clip(out, 0, 255)
    return out.astype(np.uint8)

depth_1_jpg = depth_to_uint8_for_jpg(depth_1)
depth_2_remapped_jpg = depth_to_uint8_for_jpg(depth_2_remapped)
Image.fromarray(depth_1_jpg).save(export_rgb1_jpg, quality=95)
Image.fromarray(depth_2_remapped_jpg).save(export_rgb2_jpg, quality=95)
print(f"Export JPG OK (depth_1 as JPG): {export_rgb1_jpg}")
print(f"Export JPG OK (depth_2_remapped as JPG): {export_rgb2_jpg}")

Export OK : C:\Users\mvm\open3d_vision\data\processed\depth_field_v3_bundle.npz
  depth 1811x1350, pad=16
Export JPG OK (depth_1 as JPG): C:\Users\mvm\open3d_vision\data\processed\depth_field_v3_rgb1.jpg
Export JPG OK (depth_2_remapped as JPG): C:\Users\mvm\open3d_vision\data\processed\depth_field_v3_rgb2.jpg


In [ ]:
depth_